**This is the Obsidian Section**


In [ ]:
import sys
import os
import pandas as pd
pd.set_option('display.max_colwidth', None)
# Get the path to the root directory (one level up from 'notebooks')
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the project root to the system path
if project_root not in sys.path:
    sys.path.append(project_root)

from src.parse_obsidian import iterate_files
import pandas as pd

# Folder containing Markdown files
folder_path ="../data"
# Create DataFrame
df = iterate_files(folder_path)

# Display the DataFrame
print(df)
# 239 added to 459 new total rows 699

In [ ]:
# The \ tells regex "I literally mean a bracket"
from src.dataframe_processing import remove_pasted_images
df = remove_pasted_images(df)
df[df['content'].str.contains(r'\!\[\[')].shape

In [ ]:
import pandas as pd
import ollama
import time
import re
from langdetect import detect
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
df = df[:5]
# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/words')
except LookupError:
    nltk.download('punkt')
    nltk.download('punkt_tab')
    nltk.download('words')

# --- Configuration ---
OLLAMA_MODEL = 'deepseek-r1:7b'
OLLAMA_HOST = 'http://localhost:11434'

# Initialize the Ollama client once
try:
    ollama_client = ollama.Client(host=OLLAMA_HOST)
except Exception as e:
    print(f"Error initializing Ollama client: {e}. Please ensure Ollama is running at {OLLAMA_HOST}")
    exit()

# --- Smart German Umlaut Preprocessing ---

# OPTION 1: Load German word list from file (RECOMMENDED - FAST & ACCURATE)
# Auto-downloads from https://github.com/davidak/wortliste (343,000 German words)
GERMAN_WORDS = set()

def download_german_wordlist(url='https://raw.githubusercontent.com/davidak/wortliste/master/wortliste.txt', 
                              filepath='german_words.txt'):
    """Download German word list if not already present."""
    import urllib.request
    import os
    
    if os.path.exists(filepath):
        print(f"Using existing German word list: {filepath}")
        return filepath
    
    try:
        print(f"Downloading German word list from {url}...")
        urllib.request.urlretrieve(url, filepath)
        print(f"✓ Downloaded {filepath} successfully!")
        return filepath
    except Exception as e:
        print(f"Warning: Could not download word list: {e}")
        return None

def load_german_wordlist(filepath='german_words.txt'):
    """Load German dictionary from file. Falls back to basic set if file not found."""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            words = set(line.strip().lower() for line in f if line.strip())
            print(f"✓ Loaded {len(words):,} German words from {filepath}")
            return words
    except FileNotFoundError:
        print(f"Warning: {filepath} not found. Using basic word list.")
        # Fallback to basic common words
        return {
            'vergessen', 'essen', 'wissen', 'müssen', 'lassen', 'schließen',
            'fassen', 'passen', 'hassen', 'messen', 'pressen', 'besser',
            'wasser', 'straße', 'groß', 'weiß', 'heiß', 'fuß', 'fluss',
            'schloss', 'ross', 'genossen', 'fressen', 'interessant',
            'strasse', 'gross', 'weiss', 'heiss', 'fuss', 'schliessen'
        }

# Download and load the word list
wordlist_path = download_german_wordlist()
if wordlist_path:
    GERMAN_WORDS = load_german_wordlist(wordlist_path)
else:
    GERMAN_WORDS = load_german_wordlist()  # Use fallback

# Replacement mapping
GERMAN_REPLACEMENTS = {
    'ue': 'ü',
    'oe': 'ö', 
    'ae': 'ä',
    'Ue': 'Ü',
    'Oe': 'Ö',
    'Ae': 'Ä',
}

def should_replace_ss(word: str) -> bool:
    """
    Determines if 'ss' should be replaced with 'ß' in a word.
    Returns True only if the word with 'ss' is NOT in the German dictionary.
    """
    word_lower = word.lower()
    
    # If the word with 'ss' is a known correct German word, don't replace
    if word_lower in GERMAN_WORDS:
        return False
    
    # Check if replacing 'ss' with 'ß' creates a valid word
    word_with_eszett = word_lower.replace('ss', 'ß')
    if word_with_eszett in GERMAN_WORDS:
        return True
    
    # Conservative: if neither version is in dictionary, don't replace
    return False

def smart_german_normalize(word: str, preserve_case: bool = True) -> str:
    """
    Intelligently normalizes German umlauts in a single word.
    - Checks dictionary before replacing 'ss' → 'ß'
    - Always replaces digraphs like 'ue', 'oe', 'ae'
    """
    if not word:
        return word
    
    original_word = word
    normalized = word
    
    # Handle 'ss' → 'ß' carefully
    if 'ss' in word.lower():
        if should_replace_ss(word):
            # Replace 'ss' with 'ß' while preserving case
            if 'SS' in word:
                normalized = word.replace('SS', 'ß')
            else:
                normalized = word.replace('ss', 'ß')
    
    # Always replace other digraphs (ue→ü, oe→ö, ae→ä)
    for digraph, umlaut in GERMAN_REPLACEMENTS.items():
        if digraph in normalized:
            normalized = normalized.replace(digraph, umlaut)
    
    return normalized

def normalize_german_umlauts_sentence(sentence: str) -> str:
    """
    Checks if a SINGLE sentence is German and performs smart umlaut normalization.
    Works word-by-word with dictionary checking.
    """
    if not sentence.strip():
        return sentence

    # Check if the sentence is German
    try:
        if detect(sentence) != 'de':
            return sentence
    except Exception:
        return sentence

    # Tokenize into words, preserving punctuation
    words = word_tokenize(sentence, language='german')
    
    normalized_words = []
    for word in words:
        # Only process words (not punctuation)
        if word.isalpha():
            normalized_words.append(smart_german_normalize(word))
        else:
            normalized_words.append(word)
    
    # Reconstruct sentence (simple approach - join with spaces)
    # For better punctuation handling, consider using TreebankWordDetokenizer
    result = ' '.join(normalized_words)
    
    # Fix spacing around punctuation
    result = re.sub(r'\s+([.,!?;:])', r'\1', result)
    result = re.sub(r'([.,!?;:])\s*', r'\1 ', result)
    
    return result.strip()

def preprocess_text_entry(text: str) -> str:
    """
    Splits the entry into sentences, processes each one, and rejoins them.
    This is the efficient, sentence-wise preprocessor with smart substitution.
    """
    sentences = sent_tokenize(text)
    processed_sentences = []
    
    for sentence in sentences:
        processed_sentences.append(normalize_german_umlauts_sentence(sentence))
    
    return ' '.join(processed_sentences)

# --- DeepSeek R1 Output Cleaning ---
def extract_final_answer(response_text: str) -> str:
    """
    Extracts the final answer from DeepSeek R1 output by removing reasoning tags.
    """
    cleaned = re.sub(r'<think>.*?</think>', '', response_text, flags=re.DOTALL | re.IGNORECASE)
    cleaned = re.sub(r'<reasoning>.*?</reasoning>', '', cleaned, flags=re.DOTALL | re.IGNORECASE)
    
    prefixes_to_remove = [
        r'^(Sure[,!.]?\s*)',
        r'^(Here\s+(is|are)\s+the\s+.*?:?\s*)',
        r'^(The\s+corrected\s+text\s+is:?\s*)',
        r'^(Corrected:?\s*)',
        r'^(Answer:?\s*)',
        r'^(I have corrected your text:?\s*)',
        r'^(Corrected sentence:?\s*)',
    ]
    
    for prefix in prefixes_to_remove:
        cleaned = re.sub(prefix, '', cleaned, flags=re.IGNORECASE)
    
    cleaned = cleaned.strip()
    if (cleaned.startswith('"') and cleaned.endswith('"')) or \
       (cleaned.startswith("'") and cleaned.endswith("'")):
        cleaned = cleaned[1:-1]
    
    return cleaned.strip()

# --- LLM Correction Function ---
def llm_correct_typo(text: str, client: ollama.Client) -> str:
    """
    Uses an LLM via Ollama to correct typos in a single text string.
    Filters out reasoning steps and conversational text.
    """
    system_prompt = (
        "You are a strict multilingual typo correction tool. "
        "Your task: correct spelling/typos in English, French, or German. "
        "Return ONLY the corrected sentence with no explanations, reasoning, or extra text. "
        "If there are no errors, return the original text exactly."
    )
    
    user_prompt = f"Correct any typos in this text and return ONLY the corrected version:\n\n{text}"
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    
    try:
        response = client.chat(
            model=OLLAMA_MODEL,
            messages=messages,
            options={'temperature': 0.1}
        )
        
        raw_response = response['message']['content']
        cleaned_response = extract_final_answer(raw_response)
        
        return cleaned_response
        
    except Exception as e:
        print(f"\n[ERROR] Processing text '{text[:40]}...': {e}")
        return text

# --- Create and Process the DataFrame ---
data = {
    'content': [
        "hte quick brown fox. Ich gehe zur Strasse. Der Fluegel ist rot.",
        "Ich habe das vergessen. Die Strasse ist lang.",
        "J'ai mangé une pomme. This is an English sentence.",
        "C'est un chein.",
        "Ich habe ein katz. Die Schoenheit der Natur.",
        "Wir muessen das wissen. Das Wasser ist heiss."
    ]
}


# df = df[:5]  # Uncomment to limit processing

# Step 1: Apply smart preprocessing for umlaut normalization
print("Applying smart sentence-wise preprocessing with dictionary checking...")
df['content_preprocessed'] = df['content'].apply(preprocess_text_entry)

# Step 2: Apply LLM correction to the preprocessed text
corrected_list = []
total_rows = len(df)
print(f"Starting LLM correction for {total_rows} rows using {OLLAMA_MODEL}...\n")

for index, row in df.iterrows():
    preprocessed_text = row['content_preprocessed']
    corrected_text = llm_correct_typo(preprocessed_text, ollama_client)
    corrected_list.append(corrected_text)
    
    time.sleep(0.5)
    print(f"Processed {index + 1}/{total_rows} entries...", end='\r')

df['corrected_content_LLM'] = corrected_list

# --- Final Clean Display ---
df['corrected_content_LLM'] = df['corrected_content_LLM'].str.strip()

print("\n" + "="*70)
print("FINAL RESULTS (Original --> Preprocessed --> Corrected)")
print("="*70)

original_series = df['content'].values
preprocessed_series = df['content_preprocessed'].values
corrected_series = df['corrected_content_LLM'].values

for original, preprocessed, corrected in zip(original_series, preprocessed_series, corrected_series):
    print(f"Original:      {original}")
    print(f"Preprocessed:  {preprocessed}")
    print(f"Corrected:     {corrected}")
    print("-" * 70)

print("="*70)

In [ ]:

def drop_unnecessary_rows(df):
    non_empty_df = df.query('content != ""') # filter out all the values that are empty 

    empty_df = df.query('content == ""')
    return non_empty_df


def preprocess_dataframe(df):
    non_empty_df = drop_unnecessary_rows(df)
   
    #sorting by ascending date
    non_empty_df['date'] = non_empty_df['filename'].apply(extract_date)
    df_sorted = non_empty_df.sort_values(by='date')
    return df_sorted

In [ ]:
# dataframe preprocessing 
# filter out the empty entries
df = df[df['content'].str.strip() != '']
# filtered out 1 line here

In [ ]:
from src.dataframe_processing import get_all_values_of_column
df, all_people = get_all_values_of_column(df, 'People')
#replacements ideally automatic with 
print(all_people)
# find possible , mistakes
# find likely mistakes with typos (levensthein distance)

# check all names are valid
# check for permutations assumption if the permutation is the same then the person is probably the same

# check for occurrences the assumption being that usually the name that occcurs more often is the correct spelling. 
occurrences = [4,2,1,1]
names = [ name for name,value_count in all_people]
#makeing a dict for faster lookup
all_people_dict = { name:value_count for name,value_count in all_people}
print(names)
alphabetical_permutations = [''.join(sorted(name.lower())) for name in names]
# probably when there is a typo in the permutation it is more likely to be the same person 
zip_permutations = list(zip(names, alphabetical_permutations))
# sort after the permutation
zip_permutations = sorted(zip_permutations, key=lambda x: (x[1], x[0])) # sort by permutation then name
names_subsitutions = {} # with first name being the name that gets substituted
for i in range(len(zip_permutations)-1):
    name1, perm1 = zip_permutations[i]
    for j in range(i+1, len(zip_permutations)):
        name2, perm2 = zip_permutations[j]
        # if the permutations are the same then the names are likely the same person
        if perm1 == perm2:
            print(f"Possible same person: {name1} and {name2} (permutation: {perm1})") # technically multiple collisions are possible but unlikely
            if all_people_dict[name1] > all_people_dict[name2]:
                names_subsitutions[name2] = name1

            
        else:
            break


   
    




In [ ]:
from src.ollama_stuff import check_name_typos, strip_reasoning, parse_name_groups
# Example usage with your data
if __name__ == "__main__":
    # Example name list with various issues
    test_names = [
        "John Smith", "Jon Smith", "Jane Doe", "Jahn Smith", 
        "Jane Dow", "Michael Johnson", "Mike Johnson", "sister",
        "tom tailer", "Tom Taylor", "Alex", "Alex(online)",
        "Sarah Connor", "Sara Connor", "Bob Wilson", "Robert Wilson"
    ]
    
    # Custom rules for your specific dataset
    custom_rules = """
    - "bruder" and "jan baier" refer to the same person
    - All other variations of "jan" are distinct persons
    - Sometimes the , is missing between two names like "lukaspatrick
    - Names with "(online)" annotations refer to the same person as the base name
    """
    
    # Process names
    result = check_name_typos(all_people, custom_rules=custom_rules)
    print(result)
    clean_result = strip_reasoning(result)
    parsed_groups = parse_name_groups(clean_result)
    
    print("Raw model output:")
    print(result)
    print("\nCleaned output:")
    print(clean_result)
    print("\nParsed groups:")
    for i, group in enumerate(parsed_groups, 1):
        print(f"Group {i}: {group}")

In [ ]:
from src.dataframe_processing import normalize_ratings, extract_date, drop_unnecessary_columns,check_coocurrence
df = normalize_ratings(df)
df = drop_unnecessary_columns(df)

>Parsing of the data

In [ ]:


df['Activities'].value_counts()
df['Activities2']  = df['Activities'] .apply(lambda x: x.replace(',', '') if isinstance(x, str) else x)
df['Activities2'] = df['Activities2'].apply(lambda x: x.split(' '))

df['Activities2'].value_counts()




# Apply the function to create a new datetime column
df_sorted = preprocess_dataframe(df)
df_sorted.head()




In [ ]:
# Activity analysis 
df['Activities'].value_counts()
df['Activities']  = df['Activities'] .apply(lambda x: x.replace(' ', '') if isinstance(x, str) else x)
df['Activities3'] = df['Activities'].apply(lambda x: x.split(','))

df['Activities3'] = df['Activities3'].apply(lambda x: sorted(x) if isinstance(x, list) else x)
df['Activities3'].value_counts()


# Finden von Schreibfehlern

# Gesamtliste

def convert_to_list(df, column_name):
    # Step 1: Clean strings (remove spaces) and leave floats/NaN as-is
    df[column_name] = df[column_name].apply(
        lambda x: x.replace(' ', '') if isinstance(x, str) else x
    )
    # Step 2: Split strings by comma, convert non-strings to empty lists
    df[column_name] = df[column_name].apply(
        lambda x: x.split(',') if isinstance(x, str) else []
    )
    # Step 3: Sort lists (if they exist), otherwise keep as-is
    df[column_name] = df[column_name].apply(
        lambda x: sorted(x) if isinstance(x, list) else x
    )
    return df


def normalize_list(input_list):
    input_list = [x.lower() for x in input_list]
    return input_list
def get_total_list(df, column_name):
    total_list = list(set().union(*df[column_name]))
    total_list = normalize_list(total_list)
    total_list = sorted(list(set(total_list)))
    return total_list






In [ ]:
all_activities = list(set().union(*df['Activities2']))
all_activities = [x.lower() for x in all_activities]
all_activities = sorted(list(set(all_activities)))

print(all_activities)
print(len(all_activities))
print(len(all_activities))
print(df.columns)
copy, all_activities2 = get_all_values_of_column(df, "Activities")
print(all_activities2)
print(len(all_activities2))
print(list(set(all_activities)-set(all_activities2)))

In [ ]:
# Check what dart co-occurs with
check_coocurrence(df, 'Activities3', 'Frisbee')

In [ ]:
df["Essen"].dtype
print(df["Essen"].value_counts())
_, all_activities2 = get_all_values_of_column(df, "Essen")
df = convert_to_list(df, 'Essen')
df = convert_to_list(df, 'Trinken')
all_essen = get_total_list(df, 'Essen')
print(all_essen)
print()
print(list(set(all_activities2)-set(all_essen)))

In [ ]:
import re
from src.dataframe_processing import generate_specialized_dataframe_with_counts
print(df.columns)
print(df['Trinken'].value_counts())
print(df['Essen'].value_counts())
essen_df = generate_specialized_dataframe_with_counts(df, 'Essen')

trinken_df = generate_specialized_dataframe_with_counts(df, 'Trinken')
print(essen_df.head(10))
print(trinken_df.head(10))
        



# Apply to a DataFrame where each entry is a list of items


In [ ]:
from collections import defaultdict
import re
import re
import pandas as pd
from collections import defaultdict

# --- CORE NORMALIZATION LOGIC ---

def _process_item_core(item):
    """
    Core logic to extract quantity, convert to liters, and clean the item name.
    
    Returns (standardized_name, quantity_in_liters)
    """
    item_str = str(item).strip()

    # 1. Regex to capture: (Quantity) (Optional Unit/Multiplier) (Item Name)
    # The regex targets standard quantity formats and common liquid units (L, ML, etc.).
    # Quantity: (\d*\.?\d+) 
    # Unit: (\s*[a-zA-Z]*) -> Optional space, then optional letters (for unit)
    # Item: (.*) -> Everything else is the item name (or multiplier 'x')
    match = re.match(r'^(\d*\.?\d+)\s*([a-zA-Z]*)\s*(.*)$', item_str, re.IGNORECASE)

    quantity = 1.0 # Default if no number is found
    unit = ""
    raw_name = item_str
    
    if match:
        quantity_str, unit, raw_name = match.groups()
        try:
            quantity = float(quantity_str)
        except ValueError:
             # Fallback if float conversion fails
             quantity = 1.0
             raw_name = item_str
    
    # --- 2. Standardize Unit and Quantity (Conversion to Liters) ---
    unit = unit.lower().strip()
    
    quantity_in_liters = quantity
    
    if unit in ('ml'):
        # Convert milliliters to liters
        quantity_in_liters /= 1000.0
    elif unit in ('x', '', 'stk'):
        # If the unit is 'x' (multiplier) or missing, we assume the item 
        # is a standard serving size (e.g., one can of Cola).
        # We need a defined standard size for aggregation. Let's assume 
        # a standard serving is 0.33 Liters (e.g., a small bottle/can)
        # You may need to adjust this factor based on your data!
        quantity_in_liters *= 0.33
    
    # --- 3. Clean Item Name ---
    temp_name = raw_name.strip()
    
    # Strip common prefixes/units from the name if they were left there
    unit_prefixes = ['g', 'l', 'ml', 'gr', 'lt', 'x', 'stk']
    
    for prefix in unit_prefixes:
        if temp_name.lower().startswith(prefix):
            temp_name = temp_name[len(prefix):].strip()
            continue
            
    # Final Standardization
    standardized_name = temp_name.strip().title()

    # --- 4. Final Aggregation Mapping (Unification) ---
    # Manually map common variations to a single, consistent name.
    if "Wasser" in standardized_name:
        standardized_name = "Water"
    elif "Bier" in standardized_name or "Weizen" in standardized_name or "Kilkenny" in standardized_name or "Radler" in standardized_name:
        standardized_name = "Beer/Malt"
    elif "Cola" in standardized_name or "Mate" in standardized_name:
        standardized_name = "Soda/Mate"
    elif "Apfelschorle" in standardized_name:
        standardized_name = "Apfelschorle"
    elif not standardized_name and quantity_in_liters > 0:
        # If the name is blank but we have a quantity, assign it to a category
        standardized_name = "Uncategorized Drink"

    # Skip items where the name is still empty (e.g., rows that were just empty delimiters)
    if not standardized_name:
        return "", 0.0
        
    return standardized_name, quantity_in_liters


# --- COMPATIBILITY WRAPPERS (Unchanged) ---

def normalize_item(item):
    """Placeholder for backward compatibility. Uses core logic."""
    name, quantity = _process_item_core(item)
    return f"{quantity} {name}" if name else str(item).strip()

def normalize_and_count(item):
    """Extract item name and quantity (default to 1 if no prefix). Uses core logic."""
    return _process_item_core(item)


# --- REVISED MAIN PROCESSING FUNCTION (Crucially, handles string splitting) ---

def generate_specialized_dataframe_with_counts(df, column_name):
    """
    Processes the specified DataFrame column to calculate total item counts 
    after robust extraction and unit conversion.
    """
    
    item_counts = defaultdict(float)

    # Directly iterate over the column containing the raw lists
    for raw_entry in df[column_name]:
        
        # 1. Skip NaN/Null rows
        if pd.isna(raw_entry):
            continue
            
        # 2. Extract individual items (handling list format OR single string format)
        if isinstance(raw_entry, list):
            items_to_process = raw_entry
        elif isinstance(raw_entry, str):
            # Handles the packed string format from value_counts, e.g., 
            # "[0.5LWeizen, 1LApfelschorle, 2LWasser]"
            # Strip brackets, then split by comma, and strip spaces
            items_to_process = [
                item.strip() for item in raw_entry.strip('[]').split(',') if item.strip()
            ]
        else:
            # Handle unexpected types (e.g., single floats from errors)
            continue
            
        # 3. Process each individual item
        for item in items_to_process:
            
            # Use the core normalization logic directly on the RAW item string
            name, quantity = _process_item_core(item)
            
            # Aggregate counts, skipping empty names
            if name:
                item_counts[name] += quantity

    # 4. Convert to a clean DataFrame for output
    count_df = pd.DataFrame(
        {"Item": item_counts.keys(), "Total Liters": item_counts.values()}
    ).sort_values("Total Liters", ascending=False).reset_index(drop=True)
    
    return count_df

trinken_df = generate_specialized_dataframe_with_counts(df, 'Trinken')
print(essen_df.head(10))

In [ ]:
from sentence_transformers import SentenceTransformer
from huggingface_hub import login
import torch
import torch
torch.cuda.empty_cache()
login("dummy")  # Replace with your actual Hugging Face token
# --- Setup and Device Check ---
print(f"CUDA is available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    # Define the device to use
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# --- Model Loading and Device Assignment ---
# 1. Load the model
model = SentenceTransformer("google/embeddinggemma-300m")
# 2. Move the model to the selected device (GPU if available)
model.to(device)
print(f"Model moved to {device}")

# --- Inference Data ---
query = "Which planet is known as the Red Planet?"
documents = [
    "Venus is often called Earth's twin because of its similar size and proximity.",
    "Mars, known for its reddish appearance, is often referred to as the Red Planet.",
    "Jupiter, the largest planet in our solar system, has a prominent red spot.",
    "Saturn, famous for its rings, is sometimes mistaken for the Red Planet."
]

# --- Encoding and Device Assignment for Input ---
# When using SentenceTransformer with a GPU, you generally don't need to manually
# move the *input text* (query/documents) to the GPU. The model's `encode_query`
# and `encode_document` methods handle tokenization and automatically move the
# resulting tensors to the same device the model is on (which we set with `model.to(device)`).
# We can confirm the output tensors are on the GPU.

# The `model.encode_query` and `model.encode_document` methods from SentenceTransformer
# return a NumPy array by default unless specified. We need to ensure the output
# is a PyTorch tensor and is on the correct device for the `model.similarity` call.
# The `convert_to_tensor=True` and `device=device` arguments are key here.

query_embeddings = model.encode_query(query, convert_to_tensor=True, device=device)
print("Query embedded")
document_embeddings = model.encode_document(documents, convert_to_tensor=True, device=device)

print(f"Query embeddings shape: {query_embeddings.shape}")
print(f"Document embeddings shape: {document_embeddings.shape}")

# Verify the device of the resulting tensors
print(f"Query embeddings device: {query_embeddings.device}")
print(f"Document embeddings device: {document_embeddings.device}")

# --- Compute ~/.cache/huggingface/hub/models--google--embeddinggemma-300mSimilarities ---
# 3. The `model.similarity` method now runs the computation on the GPU
# because both input tensors (`query_embeddings` and `document_embeddings`) are on the GPU.
similarities = model.similarity(query_embeddings, document_embeddings)

print("\nSimilarities (on GPU):")
print(similarities)
print(f"Similarities tensor device: {similarities.device}")

In [ ]:
from sentence_transformers import SentenceTransformer
import torch
import pandas as pd
import numpy as np
from src.embedding import classify_item

# --- Setup and Device Check (from previous code) ---
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")
print(essen_df)
# --- Model Loading and Device Assignment ---
# Assuming the model is already downloaded and cached
model = SentenceTransformer("google/embeddinggemma-300m")
model.to(device)
print(f"Model moved to {device}")

# --- 1. Define Classification Labels and Prompts ---
# Define the categories you want to classify into
def get_similarities(categories, classification_prompts, model, device, df_slice):
    category_embeddings = model.encode_document(classification_prompts, convert_to_tensor=True, device=device)
    df_classified = classify_item(
    df_slice, 
    model, 
    category_embeddings, 
    categories, 
    device, 
    BATCH_SIZE)
    return df_classified


categories = ["sweets", "meat", "vegetarian"]

# Define a descriptive prompt for each category to ensure high-quality embeddings.
# This makes the embeddings more representative of the entire category.
classification_prompts = [
    "A food item that is generally high in sugar, such as a dessert, candy, or pastry.", # sweets
    "A dish or ingredient consisting of animal flesh or poultry, like beef, chicken, or fish.", # meat
    "A food item that is made purely from plants, vegetables, fruits, or dairy, likely without any animal flesh." # vegetarian
]

# --- 2. Encode the Labels/Prompts ---
# Encode the classification prompts once and keep them on the GPU
print("Encoding classification prompts...")
category_embeddings = model.encode_document(
    classification_prompts,
    convert_to_tensor=True,
    device=device
)
# Shape will be (3, 768) for 3 categories
print(f"Category embeddings shape: {category_embeddings.shape}")

# --- Create Example DataFrame ---
data = {
    'food': [
        "Chocolate ice cream with fudge sauce",
        "Grilled salmon with roasted asparagus",
        "Spicy black bean burger patty",
        "A piece of prime rib steak",
        "Apple pie slice"
    ]
}
#df = pd.DataFrame(data)

# --- 3. Process the DataFrame in Batches ---
BATCH_SIZE = 32 # Adjust based on your GPU memory and efficiency needs
breakpoint()
# Run the classification function
df_classified = classify_item(
    essen_df, 
    model, 
    category_embeddings, 
    categories, 
    device, 
    BATCH_SIZE
)

# --- Final Result ---
print("\n--- Classified DataFrame ---")
print(df_classified)
# Free up GPU memory after the task is complete
del model
del category_embeddings


In [ ]:
from sentence_transformers import SentenceTransformer
import torch
import pandas as pd
import numpy as np

# --- Setup and Device Check (from previous code) ---
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# --- Model Loading and Device Assignment ---
# Assuming the model is already downloaded and cached
model = SentenceTransformer("google/embeddinggemma-300m")

print(f"Model moved to {device}")

# --- 1. Define Classification Labels and Prompts ---
# Define the categories you want to classify into
def get_similarities(categories, classification_prompts, model, device, df_slice):
    category_embeddings = model.encode_document(classification_prompts, convert_to_tensor=True, device=device)
    df_classified = classify_food_item(
    df_slice, 
    model, 
    category_embeddings, 
    categories, 
    device, 
    BATCH_SIZE)
    return df_classified


categories = [
    "water", 
    "cola like beverages", 
    "beer", 
    "wine", 
    "cocktails", 
    "shots", 
    "juices",
    "coffee"
]

# Define a descriptive prompt for each category
classification_prompts = [
    "A completely non-alcoholic, unflavored beverage, naturally occurring, consisting only of pure hydrogen and oxygen, with zero calories, sugar, or color.",
    "A non-alcoholic, sugary, carbonated soft drink, typically brown or dark colored, containing caffeine. Examples: Coke,Paulaner-Spezi, Pepsi, mate or generic soda.",
    "An alcoholic beverage that is brewed from fermented grains like malt and flavored with hops. It is typically carbonated and served in larger volumes like a bottle, can, or pint glass.",
    "An alcoholic beverage made from the fermentation of grape juice. It can be red, white, or rosé, and is not carbonated or mixed with other ingredients.",
    "A prepared alcoholic mixed drink, typically served in a full-sized glass, containing one or more spirits, liqueurs, or mixers .",
    "A very small, undiluted serving of a high-proof distilled alcoholic spirit, intended to be consumed quickly in one sip.",
    "A non-alcoholic beverage made by the extraction or pressing of natural fruits or vegetables. Contains natural sugars and vitamins and almost no coffeine.",
    "A coffeine rich beverage often hot and made from roasted beans or leaves not unlikely to contain milk."
]
# --- 2. Encode the Labels/Prompts ---
# Encode the classification prompts once and keep them on the GPU
print("Encoding classification prompts...")
category_embeddings = model.encode_document(
    classification_prompts,
    convert_to_tensor=True,
    device=device
)
# Shape will be (3, 768) for 3 categories
print(f"Category embeddings shape: {category_embeddings.shape}")

# --- Create Example DataFrame ---
data = {
    'food': [
        "Chocolate ice cream with fudge sauce",
        "Grilled salmon with roasted asparagus",
        "Spicy black bean burger patty",
        "A piece of prime rib steak",
        "Apple pie slice"
    ]
}
#df = pd.DataFrame(data)

# --- 3. Process the DataFrame in Batches ---
BATCH_SIZE = 32 # Adjust based on your GPU memory and efficiency needs

def classify_food_item(df, model, category_embeddings, categories, device, batch_size):
    all_classes = []
    
    # Iterate over the DataFrame in batches for efficient GPU processing
    for i in range(0, len(df), batch_size):
        # Select batch of documents (food descriptions)
        batch_documents = df['Item'].iloc[i:i + batch_size].tolist()
        
        # Encode the batch of food items
        # Use encode_document for maximum performance
        batch_embeddings = model.encode_document(
            batch_documents,
            convert_to_tensor=True,
            device=device
        )
        
        # --- 4. Calculate Similarity and Assign the Best Class ---
        # Calculate cosine similarity between the batch embeddings and the category embeddings
        # The result will be a tensor of shape (Batch Size, Number of Categories)
        # The similarity calculation happens entirely on the GPU
        similarities = model.similarity(batch_embeddings, category_embeddings)
        
        # Find the index of the highest similarity score for each food item
        # dim=1 means finding the max across the categories
        best_match_indices = torch.argmax(similarities, dim=1).cpu().numpy()
        
        # Map the index back to the category name
        batch_classes = [categories[idx] for idx in best_match_indices]
        
        all_classes.extend(batch_classes)
        
    # Add the results to the DataFrame
    df['class'] = all_classes
    return df

# Run the classification function
df_classified = classify_food_item(
    trinken_df, 
    model, 
    category_embeddings, 
    categories, 
    device, 
    BATCH_SIZE
)

# --- Final Result ---
print("\n--- Classified DataFrame ---")
print(df_classified)
# Free up GPU memory after the task is complete
del model
del category_embeddings
torch.cuda.empty_cache()

In [ ]:

from collections import defaultdict
import re
import pandas as pd

def extract_beverage_quantity(beverage_str):
    """Extract (name, quantity) from a single beverage string (e.g., '1.5L Wasser' -> ('Wasser', 1.5))."""
    if pd.isna(beverage_str):
        return []

    # Split into individual beverages (handle commas)
    beverages = [b.strip() for b in str(beverage_str).split(",")]

    extracted = []
    for bev in beverages:
     
        # Match optional quantity (with 'L'/'l') and name
        match = re.match(r'^([\d.]+)\s*(L|l)?\s*(.*)$', bev.strip())
        if match:
            quantity, _, name = match.groups()
            extracted.append((name.strip().lower(), float(quantity)))
        else:
            # No quantity (e.g., "Kaffee" -> default to 1.0)
            extracted.append((bev.strip().lower(), 1.0))
    print(extracted)
    return extracted


# Initialize counter
beverage_counts = defaultdict(float)

# Process each row in the column (replace 'your_column' with the actual column name)

for entry in df['Trinken']:
    if "bier" in str(entry).lower():
        print(entry)
    for name, quantity in extract_beverage_quantity(entry):
        beverage_counts[name] += quantity

# Convert to a sorted DataFrame
result = pd.DataFrame(
    {"Beverage": beverage_counts.keys(), "Total Liters": beverage_counts.values()}
).sort_values("Total Liters", ascending=False)


print(beverage_counts)

In [ ]:
mask = df['Trinken'].astype(str).str.lower().str.contains("weizen", na=False)
matching_indices = df[mask].index.tolist()
print(len(df[mask].index.tolist()))
print(f"Rows with 'bier': {matching_indices}")
print(df[mask]["Trinken"])

In [ ]:
df

In [ ]:
from src.visualization_personal import visualize_rating_progression, plot_boxplot_of_weekdays
df['date'] = pd.to_datetime(df['filename'].str.extract(r'(\d{2}-\d{2}-\d{4})')[0], format='%d-%m-%Y')
df_sorted = df.sort_values('date')
visualize_rating_progression(df_sorted)

In [ ]:
df['weekday'] = df['date'].dt.day_name()
df = df[df['date'] > '2025-01-01']
df = df[df['content'] != ""]
print(df.shape)
print(df.head())

plot_boxplot_of_weekdays(df)

df['weekday'].value_counts()
df['rating'].value_counts()

In [ ]:
def get_average_performance(df, column_name, total_list):
    average_dict = {x: (0,0) for x in total_list}  
    average_rating = df['rating'].mean()
    for index, row in df.iterrows():
        items = row[column_name]
        rating = row['rating']
        if isinstance(items, list):
            for item in items:
                item = item.lower()
                if item in average_dict:
                    current_sum, count = average_dict[item]
                    average_dict[item] = (current_sum + rating, count + 1)
    average_performance = {k: (v[0] / v[1] if v[1] > 0 else 0, v[1]) for k, v in average_dict.items()}
    sorted_average = [(k, v) for k, v in sorted(average_performance.items(), key=lambda item: item[1][0])]
    delta_performance = [(k, (v[0] - average_rating, v[1])) for k, v in sorted_average]
    return delta_performance
print(get_average_performance(df, 'Activities2', all_activities))

In [ ]:
all_activities
# check whether the activities contain substrings
from collections import defaultdict
from typing import List, Dict, Tuple, Set
from src.dataframe_processing import replace_and_flatten_activities

def identify_compound_replacements(all_activities: List[str]) -> Dict[str, List[str]]:
    """
    Identifies compound words in the alphabetically sorted activity list 
    using a prefix-matching strategy within starting-character groups.
    
    Args:
        all_activities: A list of all unique, cleaned, and lowercased activity strings, 
                        assumed to be alphabetically sorted.
                        
    Returns:
        A dictionary mapping compound words to a list containing its two components.
        Example: {'programmingtest': ['programming', 'test']}
    """
    if not all_activities:
        return {}

    # 1. Group activities by their starting character
    groups = defaultdict(list)
    for activity in all_activities:
        if activity: # Ensure we don't process empty strings
            groups[activity[0]].append(activity)

    replacements = {}
    
    # Convert all activities to a set for O(1) membership check (crucial for remainder check)
    activity_set: Set[str] = set(all_activities)

    # 2. Iterate through each group to perform prefix checking
    for char, group in groups.items():
        # The group is already sorted alphabetically because the original list was sorted.
        
        # Iterate through the group, treating each element as a potential prefix
        for i in range(len(group)):
            prefix = group[i]
            
            # Iterate through the remaining elements in the group (potential compounds)
            for j in range(i + 1, len(group)):
                compound = group[j]
                
                # If a split was already found for the compound, skip it 
                # (e.g., if 'soccer' already split 'soccerpractice', we don't want 'soc' to re-split it)
                if compound in replacements:
                    continue
                
                # 3. Check for Prefix Match
                if compound.startswith(prefix):
                    # Calculate the remainder (potential second word)
                    remainder = compound[len(prefix):]
                    
                    # 4. Final Validity Check
                    # The remainder must be a non-empty string AND a known, valid activity 
                    # from the master list (activity_set).
                    if remainder and remainder in activity_set:
                        # Found a valid compound!
                        replacements[compound] = [prefix, remainder]
                        
                        # Note: We do NOT break the inner loop (j) here, because 
                        # a longer prefix later in the group (j) might also apply 
                        # and could be a more accurate split, but in this specific 
                        # algorithm, we add the first valid split found.
                        
    return replacements

# --- Example Usage (Assuming you have a full, sorted list) ---

# Example data simulation (must be sorted!)
# all_activities_example = sorted([
#     "read", "reading", "readbook", "run", "running", "soccer", 
#     "soccerpractice", "practice", "work", "workout", "out"
# ])
# print(f"Example activities: {all_activities_example}")

replacements_map = identify_compound_replacements(all_activities)
replacements_map_sorted  = {k: sorted(v) for k,v in replacements_map.items()} # sorting for faster comparison


# print("\n--- Compound Replacements Map ---")
print(replacements_map_sorted)

# 5. Apply the function to the DataFrame column
# We create a new column 'modified_activities' to store the result
df['modified_activities'] = df['Activities2'].apply(
    lambda x: replace_and_flatten_activities(x, replacements_map)
)

# Expected Output Example: 
# {'readbook': ['read', 'book'], 
#  'running': ['run', 'ning'], # May occur if 'ning' is in the list
#  'soccerpractice': ['soccer', 'practice']}

In [ ]:
all_activities = list(set().union(*df['modified_activities']))
all_activities = [x.lower() for x in all_activities]
all_activities = sorted(list(set(all_activities)))
print(all_activities)

def get_average_performance(df, column_name, total_list):
    average_dict = {x: (0,0) for x in total_list}  
    average_rating = df['rating'].mean()
    for index, row in df.iterrows():
        items = row[column_name]
        rating = row['rating']
        if isinstance(items, list):
            for item in items:
                item = item.lower()
                if item in average_dict:
                    current_sum, count = average_dict[item]
                    average_dict[item] = (current_sum + rating, count + 1)
    average_performance = {k: (v[0] / v[1] if v[1] > 0 else 0, v[1]) for k, v in average_dict.items()}
    sorted_average = [(k, v) for k, v in sorted(average_performance.items(), key=lambda item: item[1][0])]
    delta_performance = [(k, (v[0] - average_rating, v[1])) for k, v in sorted_average]
    return delta_performance
print(get_average_performance(df, 'modified_activities', all_activities))

In [ ]:
from shapley_attributions import calculate_approximate_shapley_values, display_results, simple_contrast_analysis, display_contrast_results
# Example usage:
# METHOD 1: Monte Carlo Shapley (recommended, balanced accuracy/speed)
df['activities'] = df['Activities2']  # Ensure activities column is set
shapley_vals = calculate_approximate_shapley_values(df, method='monte_carlo', n_samples=5000)
results = display_results(shapley_vals, title="MONTE CARLO SHAPLEY VALUES")

# METHOD 2: Permutation Shapley (slightly different approach)
# shapley_vals = calculate_approximate_shapley_values(df, method='permutation', n_samples=500)
# results = display_results(shapley_vals, title="PERMUTATION SHAPLEY VALUES")

# METHOD 3: Regression approximation (fastest, less accurate)
# shapley_vals = calculate_approximate_shapley_values(df, method='regression')
# results = display_results(shapley_vals, title="REGRESSION-BASED SHAPLEY VALUES")

# METHOD 4: Simple contrast (fastest, easiest to interpret)
# contrast_results = simple_contrast_analysis(df)
# results = display_contrast_results(contrast_results)

In [ ]:
# Check what dart co-occurs with
dart_days = df[df['Activities'].apply(lambda x: 'Frisbee' in x if isinstance(x, list) else False)]
print(f"Days with dart: {len(dart_days)}")
print(f"Average rating with dart: {dart_days['rating'].mean():.2f}")
print(f"Average rating without dart: {df[~df.index.isin(dart_days.index)]['rating'].mean():.2f}")

# See what activities co-occur with dart
from collections import Counter
cooccur = Counter()
for activities in dart_days['Activities']:
    for act in activities:
        if act != 'Frisbee':
            cooccur[act] += 1

print(f"\nTop activities that co-occur with dart:")
for act, count in cooccur.most_common(10):
    print(f"  {act}: {count} times ({count/len(dart_days)*100:.1f}%)")

> Generating the latex formation for printing

In [ ]:
import pandas as pd



# Generate LaTeX report
def generate_latex_report(df):
    # Start LaTeX Document
    latex_doc = r"""
    \documentclass{article}
    \usepackage{graphicx}
    \begin{document}
    \title{Project Summary Report}
    \author{Generated Report}
    \date{\today}
    \maketitle
    \section*{Summary}
    """

    # Generate section per date
    for date in df['date']:
        latex_doc += f"\n\\section*{{\\textbf{{Date: {date}}}}}\n"
        filtered = df[df['date'] == date]
        for _, row in filtered.iterrows():
            if row['People'] != "":
                 latex_doc += f"\\textbf{{People:}} {row['People']}\\\n\n"
            if row['rating'] != "":
                 latex_doc += f"\\textbf{{rating:}} {row['rating']}\\\n\n"
            if row['Activities'] != "":
                 latex_doc += f"\\textbf{{Activities:}} {row['Activities']}\\\n\n"
            latex_doc += f"\\textbf{{Content:}}\n{row['content']}\\\\\n\n"
    latex_doc += f"\n\\section*{{\\textbf{{TODOS:}}}}\n"
    for date in df['date'].unique():
        filtered = df[df['date'] == date]
        for _, row in filtered.iterrows():
            if row['TODOS'] != "false":
                 if row['TODOS'] != "":
                     latex_doc += f"\\textbf{{TODOS:}} {row['TODOS']}\\\n\n"

    # End LaTeX Document
    latex_doc += "\n\\end{document}"

    # Save LaTeX file
    with open("summary_report.tex", "w") as file:
        file.write(latex_doc)

    print("LaTeX report generated as 'summary_report.tex'")

# Generate report
generate_latex_report(df)


In [ ]:
df = df[df['rating'] >= 8.6]
df.shape
df['content']

In [ ]:
import ollama
import pandas as pd
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any
import json
from tqdm import tqdm # Optional: for progress bar

# --- 1. Define the Desired Output Structure (Same as before) ---
class DiarySection(BaseModel):
    """Represents a single, self-contained, high-level topic or note from a daily diary entry."""
    section_title: str = Field(..., description="A short, descriptive title for this section.")
    topic_summary: str = Field(..., description="A complete, self-contained summary paragraph of the note's content. This will be the text used for embedding.")
    keywords: List[str] = Field(..., description="A list of 3-5 core keywords from this section to aid search.")

# Since we handle the date in the DataFrame, we don't need the outer DailySummary model 
# in the LLM output. We ask the LLM to return a list of sections directly.
class SectionsList(BaseModel):
    sections: List[DiarySection] = Field(..., description="A list of distinct, topically separate notes generated from the diary entry.")


def process_entry_for_rag(text_content: str, model_name: str = 'deepseek-r1:7b') -> List[Dict[str, Any]]:
    """
    Calls Ollama to generate structured, semantic chunks for a single diary entry.

    Returns: A list of dictionaries, where each dict is a ready-to-embed chunk.
    """
    
    # Define the output schema for the LLM
    output_schema = SectionsList.model_json_schema()

    # 1. Construct the System Prompt
    '''system_prompt = (
    "You are an expert diary analyst and a meticulous data extraction agent. "
    "Your core task is to process a single daily entry and perform **semantic chunking**. "
    "1. **Strict Separation:** Identify and strictly separate all distinct, high-level topics, events, or threads of thought. A topic is distinct if it could be separated into its own paragraph in a book (e.g., 'Work Project Update' vs. 'Personal Event' vs. 'Local News'). "
    "2. **Self-Contained Summary:** For each distinct topic, create a **complete, self-contained summary paragraph**. This summary must be fully understandable without reading the original entry or other sections. This text will be used directly for a search index."
    "3. **Tone Preservation (Optional):** Maintain the original tone (e.g., frustrating, satisfied, stressed) where relevant in the summary."
    "The final output MUST strictly adhere to the provided JSON schema.")'''
    system_prompt = (
    "You are an expert diary analyst and a meticulous data extraction agent. "
    "Your core task is to process a single daily entry and perform **semantic chunking**. "
    "1. **Strict Separation:** Identify and strictly separate all distinct, high-level topics, events, or threads of thought. "
    "2. **Self-Contained Summary:** For each distinct topic, create a **complete, self-contained summary paragraph**. This summary must be fully understandable without reading the original entry or other sections. This text will be used directly for a search index. "
    "**CRITICAL INSTRUCTION: When summarizing, you MUST preserve all proper nouns, including names of people, places, projects, and products exactly as they appear in the original text. Do not substitute them with generic terms (e.g., 'Tom' must remain 'Tom', not 'colleague').** "
    "3. **Keywords:** Ensure the keywords list is useful for search, including the preserved proper nouns. "
    "The final output MUST strictly adhere to the provided JSON schema."
)

 
    # 2. Construct the User Message
    user_content = (
        "Here is the daily diary entry text:\n"
        f"---DIARY ENTRY---\n{text_content}\n---END ENTRY---\n"
        "Please analyze the entry and generate the structured JSON output as a list of sections."
    )

    # 3. Call Ollama
    try:
        response = ollama.chat(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content}
            ],
            format=output_schema,
            options={'temperature': 0.1} # Slightly higher temp can sometimes help break bad cycles, but keep it low
        )
        
        json_output = response['message']['content']
        
        # 4. Parse and Validate the JSON Response
        sections_model = SectionsList.model_validate_json(json_output)
        
        # Convert the Pydantic model back to a list of standard dictionaries
        return [section.model_dump() for section in sections_model.sections]
    
    except Exception as e:
        # Log the error and return an empty list for this entry
        print(f"Error processing text content: '{text_content[:50]}...'. Error: {e}")
        return []

# --- 2. Iteration and Data Aggregation ---

def process_dataframe(df: pd.DataFrame, content_column: str = 'content', date_column: str = 'date') -> pd.DataFrame:
    """
    Iterates over the DataFrame and creates a new DataFrame of RAG chunks.
    
    Args:
        df: The input DataFrame.
        content_column: The name of the column containing the diary text.
        date_column: The name of the column containing the entry date.
        
    Returns: A new DataFrame with one row per semantic chunk.
    """
    
    all_rag_chunks = []
    
    # Use iterrows() with tqdm for a progress bar if the DataFrame is large
    print(f"Starting semantic chunking for {len(df)} entries...")
    for index, row in tqdm(df.iterrows(), total=len(df)):
        original_date = row[date_column]
        original_text = row[content_column]
        
        # Get the list of structured chunks for this entry
        chunks = process_entry_for_rag(original_text)
        
        # Add metadata (date and original text) to each chunk
        for chunk in chunks:
            chunk['original_date'] = original_date
            chunk['original_entry_index'] = index # Optional: useful for tracing
            all_rag_chunks.append(chunk)

    # Create the final RAG Chunking DataFrame
    rag_df = pd.DataFrame(all_rag_chunks)
    
    return rag_df

# --- 3. Example Usage with a Dummy DataFrame ---

# Create a sample DataFrame that mimics your structure
data = {
    'date': ['2025-12-14', '2025-12-15', '2025-12-16'],
    'content': [
        "Woke up early, watched the news. The local city council voted down the new park proposal, very frustrating. "
        "Later, I spent three hours completely rewriting the documentation for the 'Authentication' module at work, "
        "it was complex but I feel satisfied with the clarity now. Dinner was chicken and rice.",
        
        "Great day! Finished the Auth module documentation early. Decided to go hiking in the afternoon, "
        "the weather was perfect. Met up with Tom and we discussed the budget for the new app. "
        "He seemed hesitant about the timeline. Got a call from my mother, she's doing fine.",
        
        "The cat was sick this morning, had to take her to the vet (cost $150). Very stressful start. "
        "Had a team meeting, things are moving slowly on the data migration project. Tried a new coffee shop, "
        "the latte was excellent."
    ]
}


# Run the processing pipeline
rag_chunks_df = process_dataframe(df, content_column='content', date_column='date')

print("\n\n--- Final RAG Chunks DataFrame ---")
print(rag_chunks_df.head(10))

# The key column for your next step (Embedding) is 'topic_summary'
print("\nExample of a ready-to-embed chunk:")
print(rag_chunks_df.iloc[0]['topic_summary'])

In [ ]:
pd.set_option('display.max_colwidth', None) 
#print(rag_chunks_df.iloc[2:3]['topic_summary'])
print(rag_chunks_df.head(3))
print(df[df['date']=='15-05-2025']['content'])

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import requests
import json
from typing import List, Dict, Any
import warnings
warnings.filterwarnings("ignore")

class OllamaNewspaperRAG:
    def __init__(self, dataframe: pd.DataFrame, text_column: str, 
                 embedding_model: str = "all-MiniLM-L6-v2",
                 ollama_model: str = "llama3.2",
                 ollama_url: str = "http://localhost:11434"):
        """
        Initialize the RAG system using Ollama for LLM generation
        
        Args:
            dataframe: DataFrame containing newspaper entries
            text_column: Name of the column containing the text content
            embedding_model: Sentence transformer model for embeddings
            ollama_model: Ollama model name (e.g., 'llama3.2', 'mistral', 'codellama')
            ollama_url: Ollama server URL
        """
        self.df = dataframe.copy()
        self.text_column = text_column
        self.ollama_model = ollama_model
        self.ollama_url = ollama_url
        
        # Initialize embedding model
        print("Loading embedding model...")
        self.embedding_model = SentenceTransformer(embedding_model)
        
        # Test Ollama connection
        self.test_ollama_connection()
        
        # Create embeddings and index
        self.create_embeddings()
        
    def test_ollama_connection(self):
        """Test connection to Ollama server"""
        try:
            response = requests.get(f"{self.ollama_url}/api/tags", timeout=5)
            if response.status_code == 200:
                models = response.json()
                available_models = [model['name'] for model in models.get('models', [])]
                print(f"✅ Connected to Ollama. Available models: {available_models}")
                
                if self.ollama_model not in available_models:
                    print(f"⚠️  Model '{self.ollama_model}' not found. Available: {available_models}")
                    print(f"Run: ollama pull {self.ollama_model}")
            else:
                print("❌ Could not connect to Ollama")
        except requests.exceptions.RequestException:
            print("❌ Ollama server not running. Start it with: ollama serve")
            
    def create_embeddings(self):
        """Create embeddings for all newspaper entries and build FAISS index"""
        print("Creating embeddings for newspaper entries...")
        
        # Extract text content and create combined searchable text
        texts = []
        for idx, row in self.df.iterrows():
            # Combine multiple columns for better search
            text_parts = []
            
            # Add main content
            main_content = str(row[self.text_column]) if pd.notna(row[self.text_column]) else ""
            text_parts.append(main_content)
            
            # Add other text columns (title, summary, etc.)
            for col in self.df.columns:
                if col != self.text_column and self.df[col].dtype == 'object':
                    col_content = str(row[col]) if pd.notna(row[col]) else ""
                    if col_content and len(col_content) < 200:  # Avoid very long fields
                        text_parts.append(f"{col}: {col_content}")
            
            combined_text = " | ".join(text_parts)
            texts.append(combined_text)
        
        self.texts = texts
        
        # Generate embeddings
        embeddings = self.embedding_model.encode(texts, show_progress_bar=True)
        
        # Create FAISS index
        dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dimension)  # Inner product for similarity
        
        # Normalize embeddings for cosine similarity
        faiss.normalize_L2(embeddings)
        self.index.add(embeddings.astype('float32'))
        
        print(f"Created index with {len(texts)} entries")
        
    def retrieve_relevant_docs(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        """Retrieve most relevant documents for a query"""
        
        # Encode query
        query_embedding = self.embedding_model.encode([query])
        faiss.normalize_L2(query_embedding)
        
        # Search in index
        scores, indices = self.index.search(query_embedding.astype('float32'), top_k)
        
        # Get relevant documents
        relevant_docs = []
        for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
            if idx < len(self.df):  # Safety check
                doc_info = {
                    'content': self.df.iloc[idx][self.text_column],
                    'score': float(score),
                    'index': int(idx),
                    'metadata': {col: self.df.iloc[idx][col] for col in self.df.columns if col != self.text_column},
                    'combined_text': self.texts[idx]
                }
                relevant_docs.append(doc_info)
        
        return relevant_docs
    
    def generate_answer_with_ollama(self, query: str, relevant_docs: List[Dict[str, Any]]) -> str:
        """Generate answer using Ollama"""
        
        # Create context from relevant documents
        context_parts = []
        for i, doc in enumerate(relevant_docs[:3]):  # Use top 3 documents
            # Include metadata for richer context
            metadata_str = ""
            if doc['metadata']:
                metadata_items = [f"{k}: {v}" for k, v in doc['metadata'].items() if v and str(v).strip()]
                metadata_str = f" ({', '.join(metadata_items)})" if metadata_items else ""
            
            content = str(doc['content'])[:800]  # Limit length
            context_parts.append(f"Source {i+1}{metadata_str}:\n{content}")
        
        context = "\n\n".join(context_parts)
        
        # Create prompt
        prompt = f"""You are analyzing newspaper articles. Based on the following sources, provide a comprehensive answer to the question.

Question: {query}

Sources:
{context}

Instructions:
- Answer based only on the information provided in the sources
- If the sources don't contain enough information, say so
- Be specific and cite relevant details
- Keep your answer concise but informative

Answer:"""
        
        # Call Ollama API
        try:
            payload = {
                "model": self.ollama_model,
                "prompt": prompt,
                "stream": False,
                "options": {
                    "temperature": 0.3,
                    "top_p": 0.9,
                    "max_tokens": 500
                }
            }
            
            response = requests.post(
                f"{self.ollama_url}/api/generate",
                json=payload,
                timeout=30
            )
            
            if response.status_code == 200:
                result = response.json()
                return result.get('response', 'No response generated').strip()
            else:
                return f"Error: Ollama API returned status {response.status_code}"
                
        except requests.exceptions.RequestException as e:
            return f"Error connecting to Ollama: {str(e)}"
    
    def query(self, question: str, top_k: int = 5) -> Dict[str, Any]:
        """Main method to query the RAG system"""
        print(f"Processing query: {question}")
        
        # Retrieve relevant documents
        relevant_docs = self.retrieve_relevant_docs(question, top_k)
        
        # Generate answer using Ollama
        answer = self.generate_answer_with_ollama(question, relevant_docs)
        
        return {
            'question': question,
            'answer': answer,
            'relevant_documents': relevant_docs,
            'sources': [(doc['index'], doc['score']) for doc in relevant_docs[:3]],
            'model_used': self.ollama_model
        }
    
    def analyze_topics(self) -> Dict[str, Any]:
        """Analyze main topics across all newspaper entries"""
        sample_texts = self.df[self.text_column].dropna().sample(min(10, len(self.df))).tolist()
        combined_sample = " ".join([text[:200] for text in sample_texts])
        
        prompt = f"""Analyze these newspaper excerpts and identify the main topics/themes:

{combined_sample}

List the 5 most prominent topics with brief descriptions."""
        
        try:
            payload = {
                "model": self.ollama_model,
                "prompt": prompt,
                "stream": False
            }
            
            response = requests.post(f"{self.ollama_url}/api/generate", json=payload, timeout=30)
            if response.status_code == 200:
                return {"topics": response.json().get('response', 'No analysis available')}
        except:
            return {"topics": "Could not analyze topics"}

# Simplified version for quick setup
class SimpleOllamaRAG:
    """Simplified version using just numpy and Ollama"""
    
    def __init__(self, dataframe: pd.DataFrame, text_column: str, ollama_model: str = "llama3.2"):
        self.df = dataframe.copy()
        self.text_column = text_column
        self.ollama_model = ollama_model
        self.ollama_url = "http://localhost:11434"
        
        print("Loading embedding model...")
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.create_embeddings()
    
    def create_embeddings(self):
        texts = self.df[self.text_column].fillna("").astype(str).tolist()
        self.embeddings = self.embedding_model.encode(texts, show_progress_bar=True)
        self.texts = texts
        print(f"Indexed {len(texts)} newspaper entries")
    
    def query(self, question: str, top_k: int = 3):
        # Find relevant documents
        query_embedding = self.embedding_model.encode([question])
        similarities = np.dot(self.embeddings, query_embedding.T).flatten()
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        
        # Build context
        context_parts = []
        for i, idx in enumerate(top_indices):
            content = self.texts[idx][:600]
            context_parts.append(f"Article {i+1}: {content}")
        
        context = "\n\n".join(context_parts)
        
        # Generate answer with Ollama
        prompt = f"""Based on these newspaper articles, answer the question concisely:

Question: {question}

Articles:
{context}

Answer:"""
        
        try:
            response = requests.post(
                f"{self.ollama_url}/api/generate",
                json={"model": self.ollama_model, "prompt": prompt, "stream": False},
                timeout=30
            )
            
            if response.status_code == 200:
                answer = response.json().get('response', 'No response').strip()
            else:
                answer = "Error generating response"
        except:
            answer = "Could not connect to Ollama"
        
        return {
            'question': question,
            'answer': answer,
            'sources': top_indices.tolist(),
            'similarities': similarities[top_indices].tolist()
        }

# Example usage
def example_usage():
    # Sample newspaper data
    sample_data = {
        'title': [
            'Economic Growth Accelerates in Q3',
            'Climate Summit Reaches Historic Agreement', 
            'AI Revolution Transforms Healthcare',
            'Local Elections Show Surprising Results',
            'Tech Giants Report Record Profits'
        ],
        'content': [
            'The national economy expanded by 3.2% in the third quarter, driven by strong consumer spending and business investment. Manufacturing output increased significantly.',
            'World leaders at the climate summit agreed on ambitious carbon reduction targets. The deal includes $100 billion in funding for developing nations.',
            'Artificial intelligence applications in healthcare are showing remarkable results. New diagnostic tools can detect diseases 90% faster than traditional methods.',
            'Municipal elections revealed unexpected shifts in voter preferences. Independent candidates gained significant ground in suburban districts.',
            'Major technology companies reported exceptional quarterly earnings. Cloud computing and AI services led the growth across the sector.'
        ],
        'date': ['2024-10-15', '2024-10-16', '2024-10-17', '2024-10-18', '2024-10-19'],
        'category': ['Business', 'Environment', 'Technology', 'Politics', 'Business']
    }
    
   
    df['title'] = df['title'].fillna("")
    df['date'] = df['title'].fillna("date")
    df['category'] = df['tags'].fillna("journal")
    
    
    # Choose implementation:
    # Full-featured:
    # rag = OllamaNewspaperRAG(df, text_column='content', ollama_model='llama3.2')
    
    # Simple version:
    rag = SimpleOllamaRAG(df, text_column='content', ollama_model='deepseek-r1:7b')
    
    # Example queries
    queries = [
        "What activity gives me the most joy?",
        "What are my throughts on security",
        "Who do I meed most times on Sunday?"
    ]
    
    for query in queries:
        print(f"\n{'='*50}")
        result = rag.query(query)
        print(f"Q: {result['question']}")
        print(f"A: {result['answer']}")
        print(f"Sources: {result['sources']}")

# Setup Instructions
print("""
SETUP INSTRUCTIONS:
==================

1. Install Ollama:
   - Visit: https://ollama.ai/download
   - Or use: curl -fsSL https://ollama.ai/install.sh | sh

2. Start Ollama server:
   ollama serve

3. Pull a model (in another terminal):
   ollama pull llama3.2        # 2B model (fast, good for most tasks)
   ollama pull mistral         # 7B model (slower, higher quality)
   ollama pull codellama       # Good for technical content
   ollama pull llama3.2:1b     # 1B model (very fast)

4. Install Python dependencies:
   pip install sentence-transformers faiss-cpu pandas numpy requests

5. Run the script!

Available Models in Ollama:
- llama3.2 (2B) - Good balance of speed/quality
- llama3.2:1b - Fastest, lower quality
- mistral - Higher quality, slower
- codellama - Good for technical content
- phi3 - Microsoft model, very fast
""")

if __name__ == "__main__":
    example_usage()

In [ ]:
# store pandas to sql database
import pandas as pd
from sqlalchemy import create_engine
import psycopg2

# Create connection

DB_USER = "niklas"
DB_PASSWORD = ...
DB_HOST = "localhost"  # or your server IP
DB_PORT = "5432"       # default PostgreSQL port
DB_NAME = "daily_notes"

# Create the connection string
connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# Store DataFrame to PostgreSQL
df.to_sql(
    name='daily_notes',
    con=engine,
    if_exists='replace',  # or 'append' or 'fail'
    index=False,
    method='multi'  # faster for large datasets
)

In [ ]:
import pandas as pd

query = "SELECT * FROM daily_notes;"
df_retrieval = pd.read_sql(query, engine)

df_retrieval.shape

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ollama
import re
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

class ArticleRAG:
    def __init__(self, df: pd.DataFrame, content_column: str = 'content', model: str = 'deepseek-r1'):
        """
        Initialize the RAG system with a pandas DataFrame
        
        Args:
            df: DataFrame containing articles
            content_column: Name of the column containing article content
            model: Ollama model name (e.g., 'deepseek-r1', 'deepseek-coder')
        """
        self.df = df.copy().reset_index(drop=True)  # Reset index to ensure continuous indices
        self.content_column = content_column
        self.model = model
        self.vectorizer = None
        self.tfidf_matrix = None
        
        self._build_index()
    
    def _build_index(self):
        """Build TF-IDF index for similarity search"""
        print("Building TF-IDF index...")
        
        # Preprocess text
        texts = self.df[self.content_column].fillna('').astype(str)
        
        # Create TF-IDF vectorizer
        self.vectorizer = TfidfVectorizer(
            stop_words='english',
            max_features=10000,
            ngram_range=(1, 2),
            lowercase=True
        )
        
        # Fit and transform texts
        self.tfidf_matrix = self.vectorizer.fit_transform(texts)
        print(f"Index built with {self.tfidf_matrix.shape[0]} articles and {self.tfidf_matrix.shape[1]} features")
    
    def _semantic_search(self, query: str, top_k: int = 10) -> List[Tuple[int, float]]:
        """
        Perform semantic search using TF-IDF similarity
        
        Args:
            query: Search query
            top_k: Number of top results to return
            
        Returns:
            List of (dataframe_index, score) tuples
        """
        # Transform query
        query_vector = self.vectorizer.transform([query.lower()])
        
        # Calculate similarities
        similarities = cosine_similarity(query_vector, self.tfidf_matrix).flatten()
        
        # Get top-k results (matrix indices)
        top_matrix_indices = np.argsort(similarities)[::-1][:top_k]
        
        # Convert matrix indices to DataFrame indices
        df_indices = self.df.index.tolist()
        results = []
        
        for matrix_idx in top_matrix_indices:
            if similarities[matrix_idx] > 0 and matrix_idx < len(df_indices):
                df_idx = df_indices[matrix_idx]
                results.append((df_idx, similarities[matrix_idx]))
        
        return results
    
    def _keyword_search(self, query: str, case_sensitive: bool = False) -> List[int]:
        """
        Perform simple keyword search
        
        Args:
            query: Search query
            case_sensitive: Whether to perform case-sensitive search
            
        Returns:
            List of matching article indices
        """
        if case_sensitive:
            mask = self.df[self.content_column].str.contains(query, na=False, regex=False)
        else:
            mask = self.df[self.content_column].str.contains(query, na=False, case=False, regex=False)
        
        return self.df[mask].index.tolist()
    
    def search_articles(self, query: str, method: str = 'hybrid', top_k: int = 5) -> pd.DataFrame:
        """
        Search for articles based on query
        
        Args:
            query: Search query
            method: 'semantic', 'keyword', or 'hybrid'
            top_k: Number of results to return
            
        Returns:
            DataFrame with matching articles and scores
        """
        if method == 'keyword':
            indices = self._keyword_search(query)
            results_df = self.df.loc[indices].copy()
            results_df['relevance_score'] = 1.0  # All keyword matches get score 1.0
            
        elif method == 'semantic':
            search_results = self._semantic_search(query, top_k)
            if not search_results:
                return pd.DataFrame()
            
            indices, scores = zip(*search_results)
            results_df = self.df.loc[list(indices)].copy()
            results_df['relevance_score'] = scores
            
        else:  # hybrid
            # Combine keyword and semantic search
            keyword_indices = set(self._keyword_search(query))
            semantic_results = self._semantic_search(query, top_k * 2)
            
            # Create combined results
            all_results = {}
            
            # Add keyword matches with high scores
            for idx in keyword_indices:
                all_results[idx] = 0.8  # High score for exact matches
            
            # Add semantic matches
            for idx, score in semantic_results:
                if idx in all_results:
                    all_results[idx] = max(all_results[idx], score + 0.2)  # Boost if also keyword match
                else:
                    all_results[idx] = score
            
            # Sort by score and take top_k
            sorted_results = sorted(all_results.items(), key=lambda x: x[1], reverse=True)[:top_k]
            
            if not sorted_results:
                return pd.DataFrame()
            
            indices, scores = zip(*sorted_results)
            results_df = self.df.loc[list(indices)].copy()
            results_df['relevance_score'] = scores
        
        return results_df.sort_values('relevance_score', ascending=False)
    
    def query_with_llm(self, question: str, context_articles: int = 3) -> Dict:
        """
        Use LLM to answer questions based on retrieved articles
        
        Args:
            question: Question to answer
            context_articles: Number of top articles to use as context
            
        Returns:
            Dictionary with answer, sources, and metadata
        """
        # Search for relevant articles
        relevant_articles = self.search_articles(question, method='hybrid', top_k=context_articles)
        
        if relevant_articles.empty:
            return {
                'answer': 'No relevant articles found for your question.',
                'sources': [],
                'confidence': 0.0
            }
        
        # Prepare context
        context = ""
        sources = []
        
        for idx, row in relevant_articles.iterrows():
            article_text = row[self.content_column][:1000]  # Limit context length
            context += f"\nArticle {idx}:\n{article_text}\n"
            sources.append({
                'index': idx,
                'relevance_score': row['relevance_score'],
                'preview': article_text[:200] + "..."
            })
        
        # Create prompt
        prompt = f"""Based on the following articles, please answer the question: "{question}"

Articles:
{context}

Please provide a comprehensive answer based on the information in these articles. If the articles don't contain enough information to answer the question, please state that clearly.

Answer:"""
        
        try:
            # Call Ollama
            response = ollama.chat(
                model=self.model,
                messages=[{'role': 'user', 'content': prompt}]
            )
            
            answer = response['message']['content']
            
            return {
                'answer': answer,
                'sources': sources,
                'confidence': relevant_articles['relevance_score'].mean()
            }
            
        except Exception as e:
            return {
                'answer': f'Error calling LLM: {str(e)}',
                'sources': sources,
                'confidence': 0.0
            }

# Example usage and demo
def demo_rag_system():
    """Demo function showing how to use the RAG system"""
    
    # Create sample data (replace with your actual dataframe)
    sample_data = {
        'content': [
            "Rick and Morty is an animated science fiction sitcom that follows the adventures of an alcoholic scientist and his good-hearted but easily influenced grandson.",
            "The show Rick and Morty has gained massive popularity for its dark humor and complex storylines involving parallel dimensions.",
            "Breaking Bad is considered one of the greatest TV shows of all time, following Walter White's transformation.",
            "Game of Thrones captivated audiences worldwide with its complex political intrigue and fantasy elements.",
            "Rick Sanchez, the genius scientist from Rick and Morty, travels through different dimensions with his grandson Morty.",
            "The Office is a mockumentary sitcom that depicts the everyday work lives of office employees."
        ]
    }
    
   
    
    # Initialize RAG system
    rag = ArticleRAG(df, content_column='content', model='deepseek-r1:7b')
    
    print("=== RAG System Demo ===\n")
    
    # Example 1: Simple search
    print("1. Simple article search:")
    results = rag.search_articles("rick and morty", method='hybrid', top_k=3)
    print(f"Found {len(results)} articles mentioning 'rick and morty':")
    for idx, row in results.iterrows():
        print(f"  - Article {idx} (Score: {row['relevance_score']:.3f})")
        print(f"    Preview: {row['content'][:100]}...")
    
    print("\n" + "="*50 + "\n")
    
    # Example 2: LLM-powered query
    print("2. LLM-powered question answering:")
    response = rag.query_with_llm("What is the current opinion on rick and morty. usually only very few sentences are relevant the rest of the text is not really related to the question", context_articles=2)
    print(f"Question: What is Rick and Morty about?")
    print(f"Answer: {response['answer']}")
    print(f"Confidence: {response['confidence']:.3f}")
    print(f"Sources used: {len(response['sources'])} articles")

if __name__ == "__main__":
    demo_rag_system()

# Usage with your actual dataframe:
"""
# Load your dataframe
df = pd.read_csv('your_articles.csv')  # or however you load your data

# Initialize RAG system
rag = ArticleRAG(df, content_column='content', model='deepseek-r1')

# Search for articles
results = rag.search_articles("rick and morty", top_k=5)
print(results[['content', 'relevance_score']])

# Ask questions
response = rag.query_with_llm("What articles mention Rick and Morty?")
print(response['answer'])
"""

In [ ]:
df['content'].iloc[140]

In [ ]:
import requests

# Simple test
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "deepseek-r1:7b",
        "prompt": "Say hello",
        "stream": False
    }
)
print(response.json())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress  # For linear regression
# Rohdaten (mit leeren Werten und Kommas als Dezimaltrennern)
data = [
    [12.13, 7.02, 5],
    [1.34, 7.02, 3],
    [1.18, 7.02, 9],
    [np.nan, 9.56, 7],
    [2.09, 7.15, 9],
    [1.22, 8.5, 9],
    [1.42, 7.35, 8],
    [1.07, 7.4, 8],
    [1.55, np.nan, np.nan],
    [1.19, 10.03, 8],
    [12.4, 9.0, 9],
    [12.51, 8.56, 10],
    [np.nan, np.nan, np.nan],  # Leere Zeile "10+"
    [3.0, 9.33, np.nan],
    [12.48, 8.27, 8],
    [1.1, 9.3, 8.5],
    [12.43, 9.54, 8],
    [1.32, 10.18, 8.5],
    [1.47, 9.22, 9],
    [2.1, 8.57, 7],
    [1.05, 8.12, 7],
    [11.47, 9.27, 7],
    [2.04, np.nan, np.nan],  # "8.48?" als NaN behandelt
    [12.01, 9.49, np.nan],
    [12.35, 9.19, 10],
    [12.22, 9.04, 8],
    [12.06, 8.28, 8.5],
    [12.47, 9.01, 8.5],
    [1.35, 9.22, 9],
    [1.47, 9.29, 8],
    [1.53, 8.4, 7.5],
    [1.5, 10.24, 9],
    [1.02, np.nan, np.nan],
    [1.24, np.nan, np.nan],
    [12.08, 9.03, 7]
]

# DataFrame erstellen
df = pd.DataFrame(data, columns=['col1', 'col2', 'col3'])

# Ausgabe

df = df.dropna()


df['col1_mod'] = df['col1'].apply(lambda x: x%12 )
df['dif'] = df['col2'] - df['col1_mod']
df.loc[21, "dif"] = 9.5
df['dif2'] = (
    0.1 * df['dif'].shift(2) +  # Value two rows above
    0.3 * df['dif'].shift(1) +  # Value one row above
    0.6 * df['dif']             # Current value
)
x = df['dif2']
y = df['col3']

from scipy.stats import linregress
slope, intercept, r_value,p_value, std_err = linregress(x,y)
regression_line = slope*x + intercept
plt.scatter(x,y, color='blue', label='Data Points')
plt.plot(x,regression_line, color='red',label=f'Linear fit (R² = {r_value**2:.2f})')
plt.xlabel('sleeptime (X-axis)')
plt.ylabel('recreation (Y-axis)')
print(df)

Ideas for the conditional bandit problem:
- Necessary Context
- Indoor/outdoor activity => implict ?
- injury, weather, temperature, time, weekend/not, solo, 